# Ollama on Google Colab with Tunneling (ngrok & Cloudflare)

Ten notatnik umożliwia uruchomienie serwera **Ollama** na maszynie z GPU w Google Colab oraz wystawienie bezpiecznego portu przez tunel w celu integracji z zewnętrznymi projektami.

### Dostępne tunele:
- **ngrok**: Standardowe rozwiązanie (wymaga konta i tokenu autoryzacji).
- **Cloudflare Tunnel (trycloudflare)**: Alternatywne rozwiązanie (nie wymaga rejestracji, omija blokady zapór sieciowych typu FortiGuard/Fortinet).

In [ ]:
# 1. Zainstaluj Ollama
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os, subprocess, time

# 2. Uruchom serwer w tle (z ustawieniem OLLAMA_HOST=0.0.0.0 dla poprawnego działania tuneli)
print('Uruchamianie serwera Ollama z OLLAMA_HOST=0.0.0.0...')
os.environ['OLLAMA_HOST'] = '0.0.0.0'
subprocess.Popen(['ollama', 'serve'])
time.sleep(5)  # Chwila na inicjalizację

In [ ]:
# 3. Pobierz podstawowe modele
print('Pobieranie modelu bge-m3...')
!ollama pull bge-m3

print('Pobieranie modelu llama3.1...')
!ollama pull llama3.1

print('Pobieranie modelu nomic-embed-text...')
!ollama pull nomic-embed-text

#print('Pobieranie modelu llava...')
#!ollama pull llava:latest

print('Podstawowe modele zostały pobrane i są gotowe do użycia.')

In [ ]:
# 3.1. Pobierz dodatkowe modele (opcjonalne)

print('Pobieranie modelu gemma4:e4b...')
!ollama pull gemma4:e4b

print('Pobieranie modelu gemma4:12b...')
!ollama pull gemma4:12b

print('Pobieranie modelu llama3.2-vision...')
!ollama pull llama3.2-vision

print('Pobieranie modelu qwen2.5:14b...')
!ollama pull qwen2.5:14b

print('Pobieranie modelu bakllava...')
!ollama pull bakllava

print('Pobieranie modelu llava:13b...')
!ollama pull llava:13b

print('Wszystkie modele zostały pomyślnie pobrane!')

## Wystawienie portu serwera (Tunneling)
Uruchom **jedną** z poniższych opcji, aby uzyskać publiczny adres URL do połączenia zewnętrznego.

In [ ]:
# Opcja A: Zainstaluj ngrok i wyeksponuj port
%pip install pyngrok -q
from pyngrok import ngrok

# Jeśli masz token ngrok, odkomentuj i wpisz go poniżej:
ngrok.set_auth_token("3FAQyoX2W7ClGZsX16gwG5hhauf_ehfMzS65oAHw7uDrP8Ax")

try:
    public_url = ngrok.connect(11434)
    print("Ollama URL:", public_url.public_url)
except Exception as e:
    print("Błąd ngrok:", e)
    print("Upewnij się, że wstawiłeś authtoken z ngrok.com")

In [ ]:
# Opcja B: Zainstaluj cloudflared i wyeksponuj port (darmowy Cloudflare Tunnel)
import os, time

# Pobierz oficjalny plik wykonywalny Cloudflare Tunnel (cloudflared) jeśli nie istnieje
if not os.path.exists('cloudflared'):
    print("Pobieranie cloudflared...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

# Uruchom tunel w tle dla portu Ollamy (11434) i zapisz logi do pliku
print("Uruchamianie tunelu Cloudflare w tle...")
!nohup ./cloudflared tunnel --url http://localhost:11434 > cloudflare.log 2>&1 &

# Odczekaj 5 sekund na wygenerowanie adresu
time.sleep(5)

# Wyciągnij adres URL z logów
try:
    with open('cloudflare.log', 'r') as f:
        log_content = f.read()
    import re
    urls = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)
    if urls:
        public_url_cf = urls[-1]
        print("Ollama Cloudflare URL:", public_url_cf)
    else:
        print("⚠️ Nie znaleziono adresu URL w logach cloudflared. Zawartość logów:")
        !cat cloudflare.log | tail -n 10
except Exception as e:
    print("Błąd odczytu cloudflare.log:", e)

## Administracja i testowanie tuneli

In [ ]:
# Sprawdzenie listy pobranych modeli
!ollama list

In [ ]:
# Awaryjny restart serwera Ollama
import os, subprocess, time

# 1. Zabij poprzednie procesy ollama
!pkill ollama
time.sleep(2)

# 2. Ustaw host na 0.0.0.0 aby zezwolić na ruch z tunelu
os.environ['OLLAMA_HOST'] = '0.0.0.0'

# 3. Uruchom serwer ponownie
print('Restartowanie Ollama z OLLAMA_HOST=0.0.0.0...')
subprocess.Popen(['ollama', 'serve'])
time.sleep(5)

print('Serwer zrestartowany.')
if 'public_url' in globals():
    print('Adres ngrok:', public_url.public_url)
if 'public_url_cf' in globals():
    print('Adres Cloudflare:', public_url_cf)

In [ ]:
# Monitor stanu Ollama, ngrok i Cloudflare na żywo
import ipywidgets as widgets
from IPython.display import display, clear_output
import threading
import time
import requests
import subprocess
import os
from datetime import datetime

# Tworzenie elementów interfejsu
status_label = widgets.HTML(value="<b>Status Ollama:</b> Inicjalizacja...")
ngrok_label = widgets.HTML(value="<b>Status ngrok:</b> Sprawdzanie...")
cloudflare_label = widgets.HTML(value="<b>Status Cloudflare:</b> Sprawdzanie...")
log_output = widgets.Output(layout={'border': '1px solid #444', 'height': '200px', 'overflow_y': 'scroll'})
header = widgets.VBox([status_label, ngrok_label, cloudflare_label, widgets.HTML("<b>Logi systemowe:</b>"), log_output])
display(header)

def update_monitor():
    while True:
        # 1. Sprawdzanie Ollama
        try:
            res = requests.get("http://localhost:11434/api/tags", timeout=2)
            if res.status_code == 200:
                models = [m['name'] for m in res.json().get('models', [])]
                status_label.value = f"<b style='color:green'>Status Ollama: ONLINE</b> (Modele: {len(models)})"
            else:
                status_label.value = f"<b style='color:orange'>Status Ollama: BŁĄD {res.status_code}</b>"
        except:
            status_label.value = "<b style='color:red'>Status Ollama: OFFLINE</b>"

        # 2. Sprawdzanie ngrok
        try:
            ngrok_res = requests.get("http://localhost:4040/api/tunnels", timeout=2)
            tunnels = ngrok_res.json().get('tunnels', [])
            if tunnels:
                url = tunnels[0].get('public_url', 'Brak URL')
                ngrok_label.value = f"<b style='color:green'>Status ngrok: AKTYWNY</b> | URL: <a href='{url}' target='_blank'>{url}</a>"
            else:
                ngrok_label.value = "<b style='color:orange'>Status ngrok: Brak aktywnych tuneli</b>"
        except:
            ngrok_label.value = "<b style='color:red'>Status ngrok: OFFLINE (API nie odpowiada)</b>"

        # 3. Sprawdzanie Cloudflare
        try:
            # Sprawdzenie czy proces cloudflared działa
            pgrep_res = subprocess.run(["pgrep", "-f", "cloudflared"], capture_output=True, text=True)
            is_cf_running = bool(pgrep_res.stdout.strip())
            
            cf_url = "Brak URL"
            if os.path.exists("cloudflare.log"):
                with open("cloudflare.log", "r") as f:
                    log_content = f.read()
                import re
                urls = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)
                if urls:
                    cf_url = urls[-1]
            
            if is_cf_running and cf_url != "Brak URL":
                cloudflare_label.value = f"<b style='color:green'>Status Cloudflare: AKTYWNY</b> | URL: <a href='{cf_url}' target='_blank'>{cf_url}</a>"
            elif is_cf_running:
                cloudflare_label.value = "<b style='color:orange'>Status Cloudflare: PROCES DZIAŁA</b> (Brak URL w logach)"
            else:
                cloudflare_label.value = "<b style='color:red'>Status Cloudflare: OFFLINE (Proces nie działa)</b>"
        except Exception as e:
            cloudflare_label.value = f"<b style='color:red'>Status Cloudflare: BŁĄD ({e})</b>"

        # 4. Pobieranie logów
        with log_output:
            clear_output(wait=True)
            !ps aux | grep -E "ollama serve|ngrok|cloudflared" | grep -v grep
            print(f"\nOstatnia aktualizacja: {datetime.now().strftime('%H:%M:%S')}")

        time.sleep(5)

# Uruchomienie w tle
monitor_thread = threading.Thread(target=update_monitor, daemon=True)
monitor_thread.start()

In [ ]:
# Testowanie połączenia z Ollama przez publiczne tunele
import requests
import os
import re

print("=== Testowanie połączenia przez ngrok ===")
try:
    tunnels_res = requests.get('http://localhost:4040/api/tunnels', timeout=2)
    tunnels = tunnels_res.json().get('tunnels', [])

    if not tunnels:
        print("❌ Brak aktywnych tuneli ngrok!")
    else:
        public_url = tunnels[0]['public_url']
        print(f"🔗 Aktywny URL ngrok: {public_url}")
        print("⏳ Testowanie połączenia z Ollama przez tunel ngrok...")
        test_res = requests.get(f"{public_url}/api/tags", timeout=5)
        if test_res.status_code == 200:
            print("✅ Sukces! Tunel ngrok działa i Ollama odpowiada.")
            print("Modele:", [m['name'] for m in test_res.json().get('models', [])])
        else:
            print(f"⚠️ Tunel ngrok istnieje, ale Ollama zwróciła błąd: {test_res.status_code}")
except Exception as e:
    print(f"❌ Błąd/Brak tunelu ngrok: {e}")

print("\n=== Testowanie połączenia przez Cloudflare ===")
try:
    cf_url = None
    if os.path.exists("cloudflare.log"):
        with open("cloudflare.log", "r") as f:
            log_content = f.read()
        urls = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)
        if urls:
            cf_url = urls[-1]

    if not cf_url:
        print("❌ Brak wygenerowanego adresu Cloudflare w logach!")
    else:
        print(f"🔗 Aktywny URL Cloudflare: {cf_url}")
        print("⏳ Testowanie połączenia z Ollama przez tunel Cloudflare...")
        test_res = requests.get(f"{cf_url}/api/tags", timeout=5)
        if test_res.status_code == 200:
            print("✅ Sukces! Tunel Cloudflare działa i Ollama odpowiada.")
            print("Modele:", [m['name'] for m in test_res.json().get('models', [])])
        else:
            print(f"⚠️ Tunel Cloudflare istnieje, ale Ollama zwróciła błąd: {test_res.status_code}")
except Exception as e:
    print(f"❌ Błąd/Brak tunelu Cloudflare: {e}")